In [2]:
import math, time
import numpy as np
from numpy import linalg
import gc

def setup():
  print('Enter the number of sites (must be>= 2): ')
  sites = int(input())
  print('Enter the number of orbitals including base- ')
  orbitals = int(input())
  weights = weightGenerate(orbitals)
  print('Enter your choice for the t value- ')
  t_Value = input()
  start = time.time()
  matrix = np.zeros((int(math.pow(4,sites*orbitals)), int(math.pow(4,sites*orbitals))), np.byte)
  pairBreak(sites, orbitals, weights, matrix, t_Value)
  end = time.time()
  print(sites, "sites", orbitals, "orbitals time-", (end-start))
  return matrix

def weightGenerate(orbitals):
  weights = []
  for i in range(1,orbitals+1):
    print("Enter the weight for orbital", i)
    weights.append(float(input()))
  return weights

def pairBreak(sites, orbitals, weights, matrix, t_Value):
  for currState in range(int(math.pow(4,sites*orbitals))):
    pointer1 = ((2*sites*orbitals) - 1)
    pointer2 = pointer1 - (2*orbitals)
    while (pointer1 > 0):
      for currOrb in range (0, orbitals):
        if ((currState & 1 << pointer1) and (not (currState & 1 << pointer2))):
          # print("state- ", list(format(currState,'b').zfill(2*sites)))
          # print("pointer1- ", currState & 1 << pointer1)
          # print("pointer2- ", currState & 1 << pointer2)
          # print("result- ", ((currState & 1 << pointer1) and (not (currState & 1 << pointer2))), "\n")
          hamiltonian(pointer1, pointer2, currState, weights[currOrb], matrix, t_Value)
        if (currState & 1 << pointer1-1 and (not (currState & 1 << pointer2-1))):
          hamiltonian(pointer1-1, pointer2-1, currState, weights[currOrb], matrix, t_Value)
        if (currState & 1 << pointer1 and currState & 1 << pointer1-1):
          matrix[currState][currState] += 2                      # this is the coulomb interaction term U
        pointer1 -= 2
        pointer2 = (pointer2-2)%(2*sites*orbitals)
    pointer1 = ((2*sites*orbitals)-1-(2*orbitals))
    pointer2 = ((4*orbitals)-1)
    while (sites != 2 and (pointer1-(2*orbitals)) != pointer2):
      for currOrb in range(0, orbitals):
        if ((currState & 1 << pointer1) and (not (currState & 1 << pointer2))):
          hamiltonian(pointer1, pointer2, currState, weights[currOrb], matrix, t_Value)
        if (currState & 1 << pointer1-1 and (not (currState & 1 << pointer2-1))):
          hamiltonian(pointer1-1, pointer2-1, currState, weights[currOrb], matrix, t_Value)
        pointer1 -= 2
        pointer2 -= 2
      pointer2 += ((4*orbitals))
  #plt.figure(figsize=(10,10))
  #plt.matshow(matrix, fignum=1, cmap=plt.cm.hot)
  #axeslist = []
  #for i in range(int(math.pow(4,sites*orbitals))):
  #  axeslist.append((format(i,'b').zfill(2*sites*orbitals)))
  #plt.xticks(range(len(axeslist)), axeslist, size='large')
  #plt.yticks(range(len(axeslist)), axeslist, size='large')
  #plt.show()

def hamiltonian(firstPair, secondPair, currState, currWeight, matrix, t_Value):
  #print("first- ", firstPair)
  #print("second- ", secondPair)
  # print("\nWeight- ", currWeight * float(t_Value))
  # print("Initial-     ", list(format(currState,'b').zfill(8)))
  eigenState = currState ^ 1 << firstPair
  #print("Destruction- ", list(format(eigenState,'b').zfill(8)))
  eigenState = eigenState ^ 1 << secondPair
  #print("Creation-    ", list(format(eigenState,'b').zfill(8)), "\n")
  matrix[currState][eigenState] = currWeight * float(t_Value)
  matrix[eigenState][currState] = (1 * currWeight * float(t_Value)) # -t (or -1 infront for <a|b> to <b|a> ? this need s to be address

def calculateEigenvalues(mat):
  gc.collect()
  return linalg.eigh(mat)
  
mat = setup()
print(mat)
vals, vecs = calculateEigenvalues(mat)
vals = np.sort(vals)
#for i in vals:
#  print(i)
#print(vecs)

# Create a list to store pairs of vals and eigenState
eigenvalue_state_pairs = []

# Populate the list with vals and corresponding eigenState
for i, eigenvalue in enumerate(vals):
    currState = format(i, 'b').zfill(8)  # Assuming 8 is the number of bits in eigenState
    eigenvalue_state_pairs.append((eigenvalue, currState))

# Print vals and corresponding eigenState side by side
for eigenvalue, eigenState in eigenvalue_state_pairs:
    print(f'Eigenvalue: {eigenvalue}, EigenState: {eigenState}')

Enter the number of sites (must be>= 2): 


 2


Enter the number of orbitals including base- 


 2


Enter the weight for orbital 1


...

Enter the weight for orbital 2


 1


Enter your choice for the t value- 


 1


2 sites 2 orbitals time- 0.0016665458679199219
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 6 0 0]
 [0 0 0 ... 0 6 0]
 [0 0 0 ... 0 0 8]]
Eigenvalue: -2.472135954999579, EigenState: 00000000
Eigenvalue: -2.2360679774997902, EigenState: 00000001
Eigenvalue: -2.2360679774997894, EigenState: 00000010
Eigenvalue: -2.2360679774997894, EigenState: 00000011
Eigenvalue: -2.2360679774997885, EigenState: 00000100
Eigenvalue: -2.0000000000000004, EigenState: 00000101
Eigenvalue: -2.0, EigenState: 00000110
Eigenvalue: -2.0, EigenState: 00000111
Eigenvalue: -1.9999999999999993, EigenState: 00001000
Eigenvalue: -1.2360679774997911, EigenState: 00001001
Eigenvalue: -1.23606797749979, EigenState: 00001010
Eigenvalue: -1.2360679774997898, EigenState: 00001011
Eigenvalue: -1.2360679774997898, EigenState: 00001100
Eigenvalue: -1.2360679774997896, EigenState: 00001101
Eigenvalue: -1.2360679774997887, EigenState: 00001110
Eigenvalue: -1.2360679774997876, EigenState: 00001111
Ei

# Hubbard-Model Bitwise Hamiltonian Simulator

This repository presents an efficient, bitwise approach to constructing and diagonalizing the Hubbard-model Hamiltonian for multi-orbital lattice systems. It is designed to handle arbitrary numbers of sites and orbitals, producing the full spectrum of eigenvalues and eigenvectors without relying on heavy external dependencies.

---

## Project Overview

**Objective:**  
Build a Hamiltonian matrix for a Hubbard model defined on a user-specified lattice (number of sites × number of orbitals), then compute its full spectrum (eigenvalues and eigenvectors).

**Key Features:**  
- Bitwise encoding of electronic configurations (each configuration represented as an integer).  
- Construction of on-site Coulomb interaction terms (U) and nearest-neighbor hopping (t), including periodic boundaries, via fast bitwise operations.  
- Efficient storage in a dense matrix of size  
  $$
    4^{\text{sites} \times \text{orbitals}}
  $$  
  using double precision (`float64`) for accurate hopping amplitudes.  
- Extraction of both eigenvalues (in ascending order) and corresponding eigenvectors using a built-in diagonalization routine.

---

## Input Parameters

When the program runs, it prompts for:

1. **Number of sites** (integer ≥ 2)  
2. **Number of orbitals per site** (integer ≥ 1)  
3. **Orbital weights** (floating-point values for each orbital, used to modulate hopping strengths)  
4. **Hopping amplitude** $t$ (floating-point scalar)  

These parameters define the dimensionality and numerical values of the Hamiltonian matrix.

---

## Hamiltonian Construction Process

### 1. Dimensionality calculation  
The total number of basis states is  
$$
4^{\text{sites} \times \text{orbitals}},
$$  
since each “effective orbital” (a specific orbital type at a specific lattice site) can be empty, spin-down, spin-up, or doubly occupied.

### 2. Bitwise enumeration of states  
Each integer from 0 to $4^N - 1$ (where $N = \text{sites} \times \text{orbitals}$) represents a unique electronic configuration. The $N$ effective orbitals are mapped to pairs of bits:

- **Bit at position $2k$** encodes spin-down occupancy of the $k$th orbital/site.  
- **Bit at position $2k+1$** encodes spin-up occupancy of the same orbital/site.

This convention ensures the spin-up bit has a higher index than the spin-down bit for each orbital.

### 3. On-site interaction (U term)  
Whenever both spin bits for a given effective orbital are set (i.e., doubly occupied), a constant value $U$ (Stage 1 code uses 2.0) is added to the diagonal entry of the Hamiltonian for that state. This is summed over all doubly occupied orbitals.

### 4. Hopping terms (t term)  
For each nearest-neighbor pair of sites (including periodic boundaries), the code considers hopping between orbitals of the same type:

1. Checks if the source orbital is occupied by a specific spin and the target is vacant.  
2. Flips the corresponding bits (via XOR) in the integer representation to get the new state.  
3. Writes the hopping amplitude  
   $$
     t \times \text{orbital weight}
   $$  
   to the symmetric off-diagonal entries  
   $H_{\text{initial,final}}$ and $H_{\text{final,initial}}$.

---

## Eigenvalue and Eigenvector Calculation (Stage 1)

- The fully constructed Hamiltonian matrix is passed to a standard diagonalization routine (e.g., `numpy.linalg.eigh`).  
- The routine returns:  
  - **Eigenvalues** sorted in ascending order.  
  - **Eigenvectors** as columns of the returned matrix, each column aligned with its eigenvalue.

**Status of Stage 1: Completed.**

---

## Program Summary

- **Inputs captured:** lattice size (sites, orbitals), orbital weights, hopping amplitude $t$.  
- **Hamiltonian built:** bitwise enumeration of configurations, on-site Coulomb interactions, nearest-neighbor hopping with periodic boundaries, stored in `float64`.  
- **Diagonalization:** full spectrum of eigenvalues and eigenvectors obtained.  
- **Output logging:** all parameters, timing for construction and diagonalization, and eigenpairs are logged to `output/output1.txt`.

---

## Roadmap for Next Stages

1. **Stage 2:** Reconstruct real-space states from eigenvectors by applying creation/annihilation operators ($c^\dagger\,A\,c$) to map back to occupation configurations.  
2. **Stage 3:** Extend lattice topology to three-dimensional geometries for 4-, 5-, and 6-site clusters while preserving bitwise efficiency.  
3. **Stage 4:** Incorporate thermodynamic calculations (partition functions, specific heat) to investigate phase transitions, magnetic ordering, and other physical properties.

---

## References

- Foundational introduction to the single-band Hubbard model and its limits.  
- Exact diagonalization studies of small clusters revealing pairing phenomena and pseudogaps.  
- Algorithmic techniques for multi-orbital bitwise Hamiltonian construction.  

*(See associated literature for detailed equations and numerical benchmarks.)*  

## Next Steps to achieve :

- Find the expectation value for each state (Number of particles in each quantum state in each orbital-spin)
- Find the basis states which make most contribution to each eigenstate by ranking propbabilies of contributions


In [ ]:
## This code does the following:
# 1. Imports necessary libraries.
# 2. Defines functions for generating the Hamiltonian matrix.
# 3. Sets up user inputs and initializes the Hamiltonian matrix.
# 4. Calculates eigenvalues and eigenvectors of the Hamiltonian matrix.
# 5. Contains the main function to run the simulation.

## It is an improvement over the original code by:
# - It prints the eigenvalues and corresponding eigenvectores.

############################ IMPORTING LIBRARIES ############################

import math
import time
import numpy as np
from numpy import linalg
import gc
import sys
import os


############################# FUNCTION DEFINITIONS ############################

# Function to generate the matrix
def weightGenerate(orbitals):
  """Prompts user for orbital weights and returns them as a list."""
  weights = []
  print("Enter orbital weights:") # For clarity in the output file
  for i in range(1, orbitals + 1):
    print(f"Enter the weight for orbital {i}:")
    weights.append(float(input()))
    
  return weights

# Function to calculate the hopping term and update the Hamiltonian matrix
def hamiltonian(firstPair_bit_pos, secondPair_bit_pos, initState, weight, matrix, t_float):
  """
  Calculates the hopping term and updates the Hamiltonian matrix.
  The hopping term is t_float * weight.
  It sets H[initState, finalState] = t_float * weight
  and H[finalState, initState] = t_float * weight.
  """
  hop_value = weight * t_float
  
  # Create the final state after hopping
  # Electron removed from firstPair_bit_pos, added to secondPair_bit_pos
  finalState = initState ^ (1 << firstPair_bit_pos)
  finalState = finalState ^ (1 << secondPair_bit_pos)
  
  matrix[initState, finalState] = hop_value
  matrix[finalState, initState] = hop_value # Hamiltonian is Hermitian, hop_value is real

# Function to construct the Hamiltonian matrix
def pairBreak(sites, orbitals, weights, matrix, t_float):
  """
  Constructs the Hamiltonian matrix by calculating on-site U terms and hopping terms.
  Uses bitwise operations to represent and manipulate electron configurations.
  """
  num_total_states = 4**(sites * orbitals)
  num_config_bits = 2 * sites * orbitals # Total bits for one configuration

  for currState in range(num_total_states):
    # Initialize diagonal for this state (important if U terms are the only diagonal contribution)

    # Hopping Loop 1 (Nearest neighbors with periodic boundary conditions)
    # pointer1 scans through spin-up bit positions of orbitals
    pointer1 = num_config_bits - 1 
    # pointer2 is the corresponding spin-up bit position on the 'neighboring' site for hopping
    pointer2 = pointer1 - (2 * orbitals) 
                                        
    while (pointer1 > 0): # Iterates as long as pointer1 is a valid bit position
      # The 'currOrb_iter_idx' is the iteration index (0 to orbitals-1) for stepping through orbitals at a site.
      for currOrb_iter_idx in range(orbitals):
        if pointer1 < 0: # Safety break if pointer1 goes out of bounds
            break

        # Determine the actual orbital type (0 to M-1) based on pointer1's current position
        # Assumes bit layout: (..., site_i_orb_j_up, site_i_orb_j_dn, ...)
        # where (pointer1 // 2) is the absolute index of the (spin-up, spin-down) pair,
        # and (% orbitals) gives the type of orbital within its site.
        actual_orbital_type_idx = (pointer1 // 2) % orbitals

        # Hopping for spin-up electron from P1 to P2
        if (currState & (1 << pointer1)) and not (currState & (1 << pointer2)):
          hamiltonian(pointer1, pointer2, currState, weights[actual_orbital_type_idx], matrix, t_float)
        
        # Hopping for spin-down electron from (P1-1) to (P2-1)
        # (pointer1-1) is spin-down bit corresponding to spin-up bit pointer1
        if (pointer1 > 0 and pointer2 > 0): # Ensure P1-1 and P2-1 are valid
            if (currState & (1 << (pointer1 - 1))) and not (currState & (1 << (pointer2 - 1))):
              hamiltonian(pointer1 - 1, pointer2 - 1, currState, weights[actual_orbital_type_idx], matrix, t_float)

        # On-site Coulomb interaction (U term) for the current orbital pair (pointer1, pointer1-1)
        # Adds U (hardcoded as 2.0) if both spin-up and spin-down are occupied.
        if (pointer1 > 0): # Ensure pointer1-1 is valid
            if (currState & (1 << pointer1)) and (currState & (1 << (pointer1-1))):
              matrix[currState, currState] += 2.0 # U is 2.0

        pointer1 -= 2 # Move to the spin-up bit of the next orbital type within the same site (or next site block)
        # Update pointer2 with periodic boundary condition handling
        pointer2 = (pointer2 - 2 + num_config_bits) % num_config_bits
      
      if pointer1 < 0 and orbitals > 0 : # If P1 went negative due to large orbital count in inner loop
          break


    # Hopping Loop 2 (from original user code, active for sites != 2)
    # This loop's pointer logic is specific and preserved from the initial code.
    if sites != 2:
      pointer1_L2 = num_config_bits - 1 - (2 * orbitals)
      pointer2_L2 = (4 * orbitals) - 1 

      # Original loop condition: (sites != 2 and (pointer1-(2*orbitals)) != pointer2)
      # This implies the loop continues as long as the condition based on *initial* values of a cycle holds,
      # or based on evolving values. The original was `while (sites != 2 and (pointer1-(2*orbitals)) != pointer2):`
      # Let's interpret pointer1 and pointer2 in the condition as the current p1_L2, p2_L2.
      while (p1_L2 >= 0 and p2_L2 >=0 and (p1_L2 - (2*orbitals)) != p2_L2 ): # Check original condition + bounds
        for currOrb_iter_idx in range(orbitals):
          if p1_L2 < 0 or p2_L2 < 0: # Safety break
              break
          
          actual_orbital_type_idx_L2 = (p1_L2 // 2) % orbitals

          # Hopping for spin-up
          if (currState & (1 << p1_L2)) and not (currState & (1 << p2_L2)):
            hamiltonian(p1_L2, p2_L2, currState, weights[actual_orbital_type_idx_L2], matrix, t_float)
          # Hopping for spin-down
          if (p1_L2 > 0 and p2_L2 > 0):
              if (currState & (1 << (p1_L2-1))) and not (currState & (1 << (p2_L2-1))):
                hamiltonian(p1_L2-1, p2_L2-1, currState, weights[actual_orbital_type_idx_L2], matrix, t_float)
          
          p1_L2 -= 2
          p2_L2 -= 2
        
        if p1_L2 < 0 and orbitals > 0: # Safety break
            break
        p2_L2 += (4 * orbitals) # Update pointer2_L2 as per original logic

# Function to calculate eigenvalues and eigenvectors of the Hamiltonian matrix
def calculateEigenvalues(mat):
  """Calculates and returns eigenvalues and eigenvectors of the matrix."""
  gc.collect() # Garbage collection
  # linalg.eigh returns eigenvalues in ascending order and corresponding eigenvectors
  eigenvalues, eigenvectors = linalg.eigh(mat)
  print(f"Length of eigenvectors : {len(eigenvectors)}")
  return eigenvalues, eigenvectors

# --- Utility: Transform Eigenstate Back ---

def transform_eigenstate_back(k_eigen, eigenvectors):
    """
    Transforms the k-th eigenstate back to original basis (explicitly).
    """
    psi_original_basis = eigenvectors[:, k_eigen].copy()
    return psi_original_basis

# Function to handle user inputs and setup the Hamiltonian matrix
def setup():
  """
  Handles user inputs for sites, orbitals, weights, and t_Value.
  Initializes and returns the Hamiltonian matrix and timing info.
  """
  print('Enter the number of sites (must be >= 2):')
  sites = int(input())
  if sites < 2:
    print("Error: Number of sites must be >= 2.")
    sys.exit(1) # Exit if input is invalid

  print('Enter the number of orbitals per site (must be >= 1):') # Changed from "including base"
  orbitals = int(input())
  if orbitals < 1:
    print("Error: Number of orbitals must be >= 1.")
    sys.exit(1)

  weights = weightGenerate(orbitals)
  
  print('Enter your choice for the t value (hopping amplitude):')
  t_Value_float = float(input()) # Read t_Value as float
  
  start_time = time.time()
  
  # Initialize Hamiltonian matrix with double precision
  num_states = 4**(sites * orbitals)
  matrix = np.zeros((num_states, num_states), dtype=np.float64)
  
  pairBreak(sites, orbitals, weights, matrix, t_Value_float)
  
  end_time = time.time()
  construction_time = end_time - start_time
  
  print(f"\n{sites} sites, {orbitals} orbitals. Hopping t = {t_Value_float}")
  print(f"Hamiltonian construction time: {construction_time:.6f} seconds")
  return matrix, sites, orbitals # Return sites and orbitals for potential use later

################################# MAIN FUNCTION #############################

# Main function to run the Hubbard model simulation
def main():
  """
  Main function to run the Hubbard model simulation for Stage 1.
  Redirects output to a file.
  """
  output_dir = "output"
  if not os.path.exists(output_dir):
    os.makedirs(output_dir)
  output_filename = os.path.join(output_dir, "output1.txt")
  
  original_stdout = sys.stdout
  with open(output_filename, "w") as f:
    sys.stdout = f # Redirect stdout to the file

    print("--- Hubbard Model Bitwise Hamiltonian Simulator - Stage 1 Output ---")
    
    # Get Hamiltonian matrix and parameters
    # Input prompts and initial prints will now go to the file.
    ham_matrix, num_sites, num_orbitals = setup() 
    
    print("\nHamiltonian Matrix:")
    
    if ham_matrix.shape[0] <= 16: # Only print small matrices
        print(ham_matrix)
    else:
        print(f"(Hamiltonian matrix of shape {ham_matrix.shape} not printed due to size)")

    print("\nCalculating eigenvalues and eigenvectors...")
    start_diag_time = time.time()
    eigenvalues, eigenvectors = calculateEigenvalues(ham_matrix)
    end_diag_time = time.time()
    print(f"Diagonalization time: {end_diag_time - start_diag_time:.6f} seconds")
    
    print("\n--- Eigenvalues and Eigenvectors ---")
    if len(eigenvalues) == 0:
        print("No eigenvalues calculated (matrix might be empty).")
    else:
        print(f"Total {len(eigenvalues)} eigenpairs found.")
        print(f"Eigenvalues (sorted): {np.sort(eigenvalues)}")
    print(f"Eigenvectors shape: {eigenvectors.shape}")
  

    for i in range(len(eigenvalues)):
      val = eigenvalues[i]
      vec = eigenvectors[:, i]
      # np.array2string can make the vector print nicer, but list() is also fine.
      vec_str = np.array2string(vec, precision=6, separator=', ', threshold=num_sites*num_orbitals*4 +10) # Show all elements
      print(f"Eigenvalue: {val:.8f}, Eigenvector: {vec_str}")
      if i < len(eigenvalues) -1 :
          print("-" * 20) # Separator between eigenpairs, not for the last one.

    print("\n--- End of Stage 1 Output ---")
    
  sys.stdout = original_stdout # Restore standard output
  print(f"\nOutput successfully written to {output_filename}")
  
# Run the main function if this script is executed directly

if __name__ == "__main__":
  main()


Output successfully written to output/output1.txt


In [ ]:
#---------------------------------------------------------------------------#
# This code finds the <Neff> for each eigenstate, and prints the sorted real-space configurations.
#----------------------------------------------------------------------------#
import math
import time
import numpy as np
from numpy import linalg
import gc
import sys
import os

############################# FUNCTION DEFINITIONS ############################

def weightGenerate(orbitals):
    """
    Prompts user for orbital weights and returns them as a list.
    """
    weights = []
    print("Enter orbital weights:")
    for i in range(1, orbitals + 1):
        print(f"Enter the weight for orbital {i}:")
        weights.append(float(input()))
    return weights

def hamiltonian(firstPair_bit_pos, secondPair_bit_pos, initState, weight, matrix, t_float):
    """
    Calculates the hopping term and updates the Hamiltonian matrix.
    The hopping term is t_float * weight.
    It sets H[initState, finalState] = t_float * weight
    and H[finalState, initState] = t_float * weight.
    """
    hop_value = weight * t_float

    # Create the final state after hopping
    finalState = initState ^ (1 << firstPair_bit_pos)
    finalState = finalState ^ (1 << secondPair_bit_pos)

    matrix[initState, finalState] = hop_value
    matrix[finalState, initState] = hop_value  # Hamiltonian is Hermitian

def pairBreak(sites, orbitals, weights, matrix, t_float):
    """
    Constructs the Hamiltonian matrix by calculating on-site U terms and hopping terms.
    Uses bitwise operations to represent and manipulate electron configurations.
    """
    num_total_states = 4**(sites * orbitals)
    num_config_bits = 2 * sites * orbitals  # Total bits for one configuration

    for currState in range(num_total_states):
        # --- Loop 1: Nearest-neighbor hopping with periodic boundary conditions ---
        pointer1 = num_config_bits - 1
        pointer2 = pointer1 - (2 * orbitals)

        while pointer1 > 0:
            for currOrb_iter_idx in range(orbitals):
                if pointer1 < 0:
                    break
                # Determine which orbital this bit corresponds to
                actual_orbital_type_idx = (pointer1 // 2) % orbitals

                # Hopping for spin-up electron (bit at pointer1)
                if (currState & (1 << pointer1)) and not (currState & (1 << pointer2)):
                    hamiltonian(pointer1, pointer2, currState, weights[actual_orbital_type_idx], matrix, t_float)

                # Hopping for spin-down electron (bit at pointer1-1)
                if pointer1 > 0 and pointer2 > 0:
                    if (currState & (1 << (pointer1 - 1))) and not (currState & (1 << (pointer2 - 1))):
                        hamiltonian(pointer1 - 1, pointer2 - 1, currState, weights[actual_orbital_type_idx], matrix, t_float)

                # On-site Coulomb interaction U = 2.0 when both spin-up and spin-down are occupied
                if pointer1 > 0:
                    if (currState & (1 << pointer1)) and (currState & (1 << (pointer1 - 1))):
                        matrix[currState, currState] += 2.0

                pointer1 -= 2
                pointer2 = (pointer2 - 2 + num_config_bits) % num_config_bits

            if pointer1 < 0 and orbitals > 0:
                break

        # --- Loop 2: Additional periodic hopping for sites != 2 ---
        if sites != 2:
            pointer1_L2 = num_config_bits - 1 - (2 * orbitals)
            pointer2_L2 = (4 * orbitals) - 1

            p1_L2 = pointer1_L2
            p2_L2 = pointer2_L2
            while (p1_L2 >= 0 and p2_L2 >= 0 and (p1_L2 - (2 * orbitals)) != p2_L2):
                for currOrb_iter_idx in range(orbitals):
                    if p1_L2 < 0 or p2_L2 < 0:
                        break
                    actual_orbital_type_idx_L2 = (p1_L2 // 2) % orbitals

                    # Hopping for spin-up
                    if (currState & (1 << p1_L2)) and not (currState & (1 << p2_L2)):
                        hamiltonian(p1_L2, p2_L2, currState, weights[actual_orbital_type_idx_L2], matrix, t_float)

                    # Hopping for spin-down
                    if p1_L2 > 0 and p2_L2 > 0:
                        if (currState & (1 << (p1_L2 - 1))) and not (currState & (1 << (p2_L2 - 1))):
                            hamiltonian(p1_L2 - 1, p2_L2 - 1, currState, weights[actual_orbital_type_idx_L2], matrix, t_float)

                    p1_L2 -= 2
                    p2_L2 -= 2

                if p1_L2 < 0 and orbitals > 0:
                    break
                p2_L2 += (4 * orbitals)

def calculateEigenvalues(mat):
    """
    Calculates and returns eigenvalues and eigenvectors of the matrix.
    """
    gc.collect()  # Garbage collection
    eigenvalues, eigenvectors = linalg.eigh(mat)
    return eigenvalues, eigenvectors

def setup():
    """
    Handles user inputs for sites, orbitals, weights, and t_Value.
    Initializes and returns the Hamiltonian matrix and timing info.
    """
    print('Enter the number of sites (must be >= 2):')
    sites = int(input())
    if sites < 2:
        print("Error: Number of sites must be >= 2.", file=sys.stderr)
        sys.exit(1)

    print('Enter the number of orbitals per site (must be >= 1):')
    orbitals = int(input())
    if orbitals < 1:
        print("Error: Number of orbitals must be >= 1.", file=sys.stderr)
        sys.exit(1)

    weights = weightGenerate(orbitals)

    print('Enter your choice for the t value (hopping amplitude):')
    t_Value_float = float(input())

    start_time = time.time()

    num_states = 4**(sites * orbitals)
    matrix = np.zeros((num_states, num_states), dtype=np.float64)

    pairBreak(sites, orbitals, weights, matrix, t_Value_float)

    end_time = time.time()
    construction_time = end_time - start_time

    print(f"\n{sites} sites, {orbitals} orbitals. Hopping t = {t_Value_float}")
    print(f"Hamiltonian construction time: {construction_time:.6f} seconds")
    return matrix, sites, orbitals

# --- STAGE 2: Eigenstate Analysis (no average occupations) ---

def analyze_eigenstates_stage2(eigenvalues, eigenvectors, num_sites, num_orbitals_per_site, f_out):
    """
    Analyzes eigenstates: calculates total particle number <N>
    and identifies sorted real-space configurations. Writes analysis to f_out.
    """
    f_out.write("\n\n--- Stage 2: Real-Space Occupation Analysis (No Averages) ---\n")
    num_basis_states = eigenvectors.shape[0]

    # Precompute popcount (number of 1 bits) for each basis index
    basis_state_configs_int = np.arange(num_basis_states)
    popcounts = np.array([bin(j).count('1') for j in basis_state_configs_int])

    # Probabilities |alpha_{j,k}|^2 for basis j in eigenstate k
    V_squared = eigenvectors ** 2

    num_total_config_bits = 2 * num_sites * num_orbitals_per_site

    for k_eigen in range(len(eigenvalues)):
        eigenvalue = eigenvalues[k_eigen]
        f_out.write(f"\nEigenstate {k_eigen + 1} (Energy: {eigenvalue:.8f}):\n")

        # --- Calculate Total Particle Number <N>_k ---
        probs_k = V_squared[:, k_eigen]  # Probabilities for current eigenstate
        total_particle_number = float(np.dot(probs_k, popcounts))
        f_out.write(f"  Calculated total particle number for this eigenstate: {total_particle_number:.6f}\n")

        # --- Full Sorted Real-Space Configurations ---
        f_out.write("  Sorted real-space configurations (showing Top 10 and Bottom 10):\n")
        sorted_indices_desc = np.argsort(probs_k)[::-1]  # Descending order
        num_to_show = 10

        # Top 10 configurations
        f_out.write("  -- Top 10 configurations --\n")
        for i in range(min(num_to_show, num_basis_states)):
            basis_idx = sorted_indices_desc[i]
            prob = probs_k[basis_idx]
            binary_representation = format(basis_idx, f'0{num_total_config_bits}b')
            f_out.write(
                f"    Rank {i + 1}: Config Index {basis_idx} (Binary: {binary_representation}): "
                f"Probability = {prob:.6f}\n"
            )

        # Bottom 10 configurations
        f_out.write("  -- Bottom 10 configurations --\n")
        for i in range(1, min(num_to_show, num_basis_states) + 1):
            basis_idx = sorted_indices_desc[-i]
            prob = probs_k[basis_idx]
            binary_representation = format(basis_idx, f'0{num_total_config_bits}b')
            f_out.write(
                f"    Rank -{i}: Config Index {basis_idx} (Binary: {binary_representation}): "
                f"Probability = {prob:.6f}\n"
            )

        f_out.write("-" * 30 + "\n")

    f_out.write("\n--- End of Stage 2 Analysis ---\n")

# --- MAIN FUNCTION ---------------------

def main():
    output_dir = "output"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    original_stdout = sys.stdout  # Save original stdout

    # --- Stage 1 Execution ---
    output_filename_s1 = os.path.join(output_dir, "output1.txt")
    ham_matrix, eigenvalues, eigenvectors = None, None, None
    num_sites, num_orbitals_per_site = None, None

    with open(output_filename_s1, "w") as f_s1:
        sys.stdout = f_s1  # Redirect stdout to file for Stage 1

        print("--- Hubbard Model Bitwise Hamiltonian Simulator - Stage 1 Output ---")
        ham_matrix, num_sites, num_orbitals_per_site = setup()

        print("\nHamiltonian Matrix:")
        if ham_matrix.shape[0] <= 16:  # Print only small matrices
            np.savetxt(sys.stdout, ham_matrix, fmt='%.3f')
        elif ham_matrix.shape[0] > 0:
            print(f"(Hamiltonian matrix of shape {ham_matrix.shape} not printed due to size)")
        else:
            print("(Hamiltonian matrix is empty)")

        if ham_matrix.shape[0] > 0:
            print("\nCalculating eigenvalues and eigenvectors...")
            start_diag_time = time.time()
            eigenvalues, eigenvectors = calculateEigenvalues(ham_matrix)
            end_diag_time = time.time()
            print(f"Diagonalization time: {end_diag_time - start_diag_time:.6f} seconds")

            print("\n--- Eigenvalues and Eigenvectors (Stage 1) ---")
            if eigenvalues is None or len(eigenvalues) == 0:
                print("No eigenvalues calculated.")
            else:
                print(f"Total {len(eigenvalues)} eigenpairs found.")
                for i in range(len(eigenvalues)):
                    val = eigenvalues[i]
                    vec = eigenvectors[:, i]
                    # Threshold to force full vector printing for small-to-medium sizes
                    threshold_print = num_sites * num_orbitals_per_site * 8
                    if threshold_print == 0:
                        threshold_print = 10

                    vec_str = np.array2string(
                        vec,
                        precision=6,
                        separator=', ',
                        threshold=max(10, ham_matrix.shape[0] + 1)
                    )
                    print(f"Eigenvalue: {val:.8f}, Eigenvector: {vec_str}")
                    if i < len(eigenvalues) - 1:
                        print("-" * 20)
        else:
            print("\nSkipping eigenvalue calculation due to empty Hamiltonian.")
            eigenvalues, eigenvectors = np.array([]), np.empty((0, 0))

        print("\n--- End of Stage 1 Output ---")

    sys.stdout = original_stdout  
    print(f"\nOutput for Stage 1 successfully written to {output_filename_s1}")

    # --- Stage 2 Execution ---
    if eigenvalues is not None and len(eigenvalues) > 0 and \
       num_sites is not None and num_orbitals_per_site is not None:
        output_filename_s2 = os.path.join(output_dir, "output2.txt")
        with open(output_filename_s2, "w") as f_s2:
            analyze_eigenstates_stage2(
                eigenvalues,
                eigenvectors,
                num_sites,
                num_orbitals_per_site,
                f_s2
            )
        print(f"Output for Stage 2 successfully written to {output_filename_s2}")
    else:
        print("\nSkipping Stage 2 analysis due to missing results or empty Hamiltonian.")

if __name__ == "__main__":
    main()



Output for Stage 1 successfully written to output/output1.txt
Output for Stage 2 successfully written to output/output2.txt


In [ ]:
import os, sys, time, math, gc
import numpy as np
from numpy import linalg as LA

# ----------------------------------------------------------------------
# Geometry (4-site tetrahedron)
# ----------------------------------------------------------------------
#  sites: 0-1-2 are the base triangle, 3 is the apex.
#  edges labelled by type so we can use different hoppings t_base, t_vert
EDGES = [
    (0, 1, "base"),
    (1, 2, "base"),
    (2, 0, "base"),
    (0, 3, "vert"),
    (1, 3, "vert"),
    (2, 3, "vert"),
]

# ----------------------------------------------------------------------
# Bit-position helper
# ----------------------------------------------------------------------
def bitpos(site, orb, spin, n_orb):
    """
    Return bit position of (site, orbital, spin)
    spin = 0 (down) or 1 (up)
    """
    return 2 * (site * n_orb + orb) + spin


# ----------------------------------------------------------------------
# Build the Hamiltonian
# ----------------------------------------------------------------------
def build_hamiltonian(n_orb, w_orb, t_base, t_vert, U):
    """
    Returns dense Hamiltonian H (dim x dim) for the tetrahedron.
      n_orb : orbitals per site
      w_orb : list of orbital weights (length n_orb)
    """
    n_sites = 4
    dim     = 4 ** (n_sites * n_orb)       # number of basis states
    H       = np.zeros((dim, dim), dtype=np.float64)

    # edge-type to hopping amplitude
    def hop(e_type):
        return t_base if e_type == "base" else t_vert

    # loop over every basis configuration
    for state in range(dim):

        # -------- diagonal U term --------
        diag_val = 0.0
        for s in range(n_sites):
            for o in range(n_orb):
                down_b = bitpos(s, o, 0, n_orb)
                up_b   = bitpos(s, o, 1, n_orb)
                if (state >> down_b) & 1 and (state >> up_b) & 1:
                    diag_val += U
        if diag_val:
            H[state, state] += diag_val

        # -------- off-diagonal hopping ----
        for s_i, s_j, e_type in EDGES:
            t_hop = hop(e_type)
            for o in range(n_orb):
                w = w_orb[o]
                for spin in (0, 1):
                    bi = bitpos(s_i, o, spin, n_orb)
                    bj = bitpos(s_j, o, spin, n_orb)
                    if (state >> bi) & 1 and not (state >> bj) & 1:
                        new_state = state ^ (1 << bi) ^ (1 << bj)
                        val = -w * t_hop
                        H[state, new_state] += val
                        H[new_state, state] += val   # Hermitian symmetry
    return H


# ----------------------------------------------------------------------
# Prompt user for inputs
# ----------------------------------------------------------------------
def user_inputs():
    print("Number of orbitals per site (M >= 1): ", end="")
    M = int(input().strip())
    if M < 1:
        sys.exit("M must be >= 1")

    weights = []
    print("Enter orbital weights w_0 .. w_{M-1}:")
    for o in range(M):
        print(f"  w_{o} = ", end="")
        weights.append(float(input().strip()))

    print("Base-edge hopping   t  = ", end="")
    t_base = float(input().strip())
    print("Vertical-edge hop   t' = ", end="")
    t_vert = float(input().strip())
    print("On-site Coulomb     U  = ", end="")
    U = float(input().strip())

    return M, weights, t_base, t_vert, U


# ----------------------------------------------------------------------
# Redirect all stdout to file
# ----------------------------------------------------------------------
def redirect_stdout(path):
    """
    Context manager: with redirect_stdout("file.txt"): print("hello")
    """
    class Redirector:
        def __enter__(self):
            self._old = sys.stdout
            self._fh  = open(path, "w")
            sys.stdout = self._fh
        def __exit__(self, exc_type, exc, tb):
            sys.stdout.flush()
            sys.stdout.close()
            sys.stdout = self._old
    return Redirector()


# ----------------------------------------------------------------------
# Main
# ----------------------------------------------------------------------
def main():

    # -------- make output directory / file path ----------
    out_dir = "output"
    os.makedirs(out_dir, exist_ok=True)
    out_file = os.path.join(out_dir, "tetra_stage1.txt")

    # -------- redirect everything to the file ------------
    with redirect_stdout(out_file):

        print("--- 4-site Tetrahedral Hubbard Cluster ---\n")

        # ---- user input ----
        M, w_orb, t_base, t_vert, U = user_inputs()

        print("\nParameters:")
        print(f"  M          = {M}")
        print(f"  weights    = {w_orb}")
        print(f"  t_base     = {t_base}")
        print(f"  t_vert     = {t_vert}")
        print(f"  U          = {U}\n")

        # ---- build Hamiltonian ----
        print("Building Hamiltonian …")
        t0 = time.time()
        H  = build_hamiltonian(M, w_orb, t_base, t_vert, U)
        t1 = time.time()
        dim = H.shape[0]
        print(f"Done. dim = {dim}   build time = {t1 - t0:.2f} s")

        # ---- diagonalise ----
        print("Diagonalising …")
        t2 = time.time()
        evals, evecs = LA.eigh(H)
        t3 = time.time()
        print(f"Diagonalisation finished  ({t3 - t2:.2f} s)\n")

        # ---- show spectrum ----
        print("Eigenvalues:")
        for k, ev in enumerate(evals):
            print(f"{k:4d}   {ev: .8f}")
        print()
        
        
        print("\n--- End of log ---")

    # ---------- back on console ----------
    print(f"All output written to '{out_file}'")

# ----------------------------------------------------------------------
if __name__ == "__main__":
    main()

All output written to 'output/tetra_stage1.txt'
